# آماده‌سازی داده‌ها – فاز سوم

هدف این بخش، آماده‌سازی داده‌های iNaturalist برای تحلیل‌های آماری، پردازش تصویر و مدل‌سازی است.

### ورودی‌ها
- داده‌های پرندگان
- داده‌های پستانداران
- داده‌های خزندگان
- داده‌های دوزیستان
- تصاویر ذخیره‌شده بر اساس UUID

### مراحل پردازش
1. ادغام داده‌های چهار گروه جانوری
2. بررسی ساختار و سازگاری ستون‌ها
3. شناسایی تصاویر مشابه با شباهت بیش از ۹۰٪
4. جداسازی تصاویر تکراری از مجموعه پردازش‌شده
5. بررسی و گزارش توزیع ویژگی‌ها

### خروجی‌ها
- دیتاست یکپارچه و آماده تحلیل
- گزارش کیفیت و ساختار داده
- گزارش تصاویر مشابه
- تصاویر تکراری شناسایی‌شده

داده‌های خام و تصاویر اصلی بدون تغییر باقی می‌مانند.

### ابزارها
Python، Pandas، NumPy، Pillow و ImageHash

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.preprocessing import *
import pandas as pd

In [2]:
observations = load_observations()

print("تعداد مشاهدات:", len(observations))
print("تعداد ستون‌ها:", len(observations.columns))

display(observations.head())

تعداد مشاهدات: 28819
تعداد ستون‌ها: 17


,id,uuid,observed_on,time_observed_at,user_id,quality_grade,url,image_url,latitude,longitude,taxon_geoprivacy,common_name,taxon_class_name,taxon_order_name,taxon_family_name,location_is_synthetic,taxon_group
0,2641775,1e534ce8-9b9c-4383-9d86-130c1f00435a,2016-01-28,2023-02-08 15:53:35.059001+00:00,151218,research,http://www.inaturalist.org/observations/2641775,https://inaturalist-open-data.s3.amazonaws.com...,27.253032,52.998664,open,Variable Toad,Amphibia,Anura,Bufonidae,False,amphibians
1,2649573,e1b723e0-de9b-454b-adb8-7c65d52c7b17,2016-02-02,2023-02-08 15:53:35.059001+00:00,151218,research,http://www.inaturalist.org/observations/2649573,https://inaturalist-open-data.s3.amazonaws.com...,30.543194,45.433187,obscured,Eichwald's Toad,Amphibia,Anura,Bufonidae,True,amphibians
2,3575527,3b158e51-582a-4a9b-8d2b-aaab1c2cd3cd,2016-03-26,2022-03-22 22:59:53.211538+00:00,151218,research,http://www.inaturalist.org/observations/3575527,https://inaturalist-open-data.s3.amazonaws.com...,37.014323,55.073803,open,Eastern Tree Frog,Amphibia,Anura,Hylidae,False,amphibians
3,3627379,c7749c73-b7a7-41b5-9cfb-91ec8fa830e1,2016-03-29,2022-03-22 22:59:53.211538+00:00,151218,research,http://www.inaturalist.org/observations/3627379,https://inaturalist-open-data.s3.amazonaws.com...,36.998688,55.064743,open,Eastern Tree Frog,Amphibia,Anura,Hylidae,False,amphibians
4,3720671,e9c80fe1-a1f5-49c5-bcb9-0fa3cf6bb1e7,2016-04-14,2016-04-14 10:45:00.000000+00:00,151218,research,http://www.inaturalist.org/observations/3720671,https://inaturalist-open-data.s3.amazonaws.com...,36.992558,55.057177,open,Eastern Tree Frog,Amphibia,Anura,Hylidae,False,amphibians


In [3]:
display(
    observations["taxon_group"]
    .value_counts()
    .rename_axis("گروه جانوری")
    .to_frame("تعداد")
)

,تعداد
گروه جانوری,
birds,19453
reptiles,4370
mammals,3351
amphibians,1645


In [4]:
print("تعداد UUID:", observations["uuid"].nunique())
print("تعداد UUID تکراری:", observations["uuid"].duplicated().sum())

print(
    "تعداد مشاهدات بدون UUID:",
    observations["uuid"].isna().sum()
)

تعداد UUID: 28819
تعداد UUID تکراری: 0
تعداد مشاهدات بدون UUID: 0


In [5]:
image_paths = [
    path
    for path in IMAGE_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
]

print("تعداد تصاویر اولیه:", len(image_paths))

تعداد تصاویر اولیه: 28926


In [6]:
kept_images, removed_images = find_duplicate_images(
    image_paths,
    threshold=0.90
)

print("تصاویر اولیه:", len(image_paths))
print("تصاویر باقی‌مانده:", len(kept_images))
print("تصاویر حذف‌شده:", len(removed_images))

تصاویر اولیه: 28926
تصاویر باقی‌مانده: 28135
تصاویر حذف‌شده: 791


هیچ وقت دیگه تو این پروژه از الگوریتم با زمان توان 2 استفاده نکن !

In [7]:
for path in removed_images[:10]:
    print(path.name)

5022afa8-7251-4932-a85a-549fc387b93a.jpeg
58b43f43-b99b-43de-9cd4-05b84ba35721.jpeg
b54ab00e-1169-46d3-8fc5-cc59ec963dd1.jpg
c35e0cfa-7b96-491b-8406-ed8089ea6cea.jpeg
ca7d9be2-c867-4c90-b2e6-fb84c8156cc4.jpeg
d29c850b-77f6-4f87-bde3-da82d2a32fce.jpg
e5872606-1d1f-4390-89e4-183883ad05e3.jpg
fe0b7fc5-ade2-4edd-8b8a-b4421b035ab5.jpeg
028c9829-34ec-41f7-8763-cd4c641b4949.jpg
05b9ca42-5a8f-41fd-893c-ed1e5b1f4392.jpg


In [8]:
species = build_species_table(observations)

print("تعداد گونه‌ها:", len(species))
display(species.head(10))

تعداد گونه‌ها: 1107


,species_id,common_name,taxon_class_name,taxon_order_name,taxon_family_name
0,1,Variable Toad,Amphibia,Anura,Bufonidae
1,2,Eichwald's Toad,Amphibia,Anura,Bufonidae
2,3,Eastern Tree Frog,Amphibia,Anura,Hylidae
3,4,Taurus Frog,Amphibia,Anura,Ranidae
4,5,Persian Mountain Salamander,Amphibia,Caudata,Hynobiidae
5,6,Lorestan Newt,Amphibia,Caudata,Salamandridae
6,7,Hyrcanian Wood Frog,Amphibia,Anura,Ranidae
7,8,Lemon-yellow Tree Frog,Amphibia,Anura,Hylidae
8,9,Green Toad,Amphibia,Anura,Bufonidae
9,10,Southern Crested Newt,Amphibia,Caudata,Salamandridae


In [9]:
observations = add_species_id(observations, species)

print("تعداد مشاهدات:", len(observations))
print("تعداد ستون‌ها:", len(observations.columns))

display(
    observations[
        ["uuid", "common_name", "species_id", "observed_on", "user_id"]
    ].head()
)

تعداد مشاهدات: 28819
تعداد ستون‌ها: 18


,uuid,common_name,species_id,observed_on,user_id
0,1e534ce8-9b9c-4383-9d86-130c1f00435a,Variable Toad,1,2016-01-28,151218
1,e1b723e0-de9b-454b-adb8-7c65d52c7b17,Eichwald's Toad,2,2016-02-02,151218
2,3b158e51-582a-4a9b-8d2b-aaab1c2cd3cd,Eastern Tree Frog,3,2016-03-26,151218
3,c7749c73-b7a7-41b5-9cfb-91ec8fa830e1,Eastern Tree Frog,3,2016-03-29,151218
4,e9c80fe1-a1f5-49c5-bcb9-0fa3cf6bb1e7,Eastern Tree Frog,3,2016-04-14,151218


In [10]:
images = build_images_table(kept_images, observations)

print("تعداد تصاویر نهایی:", len(images))
display(images.head())


تعداد تصاویر نهایی: 28132


,image_id,observation_uuid,image_path,animal_present
0,001cc818-ac3e-4750-979d-4d2d9d8358f7,001cc818-ac3e-4750-979d-4d2d9d8358f7,data\raw\images\amphibians\001cc818-ac3e-4750-...,<NA>
1,0096e64b-71cf-42d9-b8ad-519bcacd14a8,0096e64b-71cf-42d9-b8ad-519bcacd14a8,data\raw\images\amphibians\0096e64b-71cf-42d9-...,<NA>
2,01057a20-6a12-4d43-8a0d-edaead5284e5,01057a20-6a12-4d43-8a0d-edaead5284e5,data\raw\images\amphibians\01057a20-6a12-4d43-...,<NA>
3,014f362c-2e4e-4e51-aa4e-76b6d2315a32,014f362c-2e4e-4e51-aa4e-76b6d2315a32,data\raw\images\amphibians\014f362c-2e4e-4e51-...,<NA>
4,0162c51c-4627-417c-b6f2-b75f3164ba69,0162c51c-4627-417c-b6f2-b75f3164ba69,data\raw\images\amphibians\0162c51c-4627-417c-...,<NA>


In [12]:
# UUID مشاهداتی که تصویر دارند
image_uuids = set(images["observation_uuid"])

# حذف مشاهدات بدون تصویر
before = len(observations)

observations = observations[
    observations["uuid"].isin(image_uuids)
].copy()

after = len(observations)

print("مشاهدات اولیه:", before)
print("مشاهدات حذف‌شده:", before - after)
print("مشاهدات باقی‌مانده:", after)

مشاهدات اولیه: 28819
مشاهدات حذف‌شده: 687
مشاهدات باقی‌مانده: 28132


اختلاف تصاویر حذف شده مرحله حذف تصاویر مشابه و این مرحله به علت تصاویر کاملا یکسان حذف شده در فاز 1 هست که از دیتا بیس حذف شدن ولی تصویرشون در دایرکتوری موجود موند. 

In [18]:
%pip install pyarrow

  Using cached pyarrow-26.0.1-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
   ---------------------------------------- 0.0/29.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/29.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/29.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/29.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/29.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/29.0 MB ? eta -:--:--
    --------------------------------------- 0.5/29.0 MB 436.9 kB/s eta 0:01:06
    --------------------------------------- 0.5/29.0 MB 436.9 kB/s eta 0:01:06
    --------------------------------------- 0.5/29.0 MB 436.9 kB/s eta 0:01:06
   - -------------------------------------- 0.8/29.0 MB 431.7 kB/s eta 0:01:06
   - -------------------------------------- 0.8/29.0 MB 431.7 kB/s eta 0:01:06
   - -------------------------------------- 0.8/29.0 MB 431.7 kB/s eta 0:01:06
   - ---------------------------


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

observations.to_parquet(
    PROCESSED_DIR / "observations.parquet",
    index=False
)

species.to_parquet(
    PROCESSED_DIR / "species.parquet",
    index=False
)

images.to_parquet(
    PROCESSED_DIR / "images.parquet",
    index=False
)

print("سه جدول با موفقیت ذخیره شدند.")

سه جدول با موفقیت ذخیره شدند.


In [20]:
report = pd.DataFrame({
    "جدول": ["Observations", "Species", "Images"],
    "تعداد ردیف": [
        len(observations),
        len(species),
        len(images)
    ],
    "تعداد ستون": [
        len(observations.columns),
        len(species.columns),
        len(images.columns)
    ]
})

display(report)

,جدول,تعداد ردیف,تعداد ستون
0,Observations,28132,18
1,Species,1107,5
2,Images,28132,4


```markdown
# آدرس فایل‌های داده

تمام جدول‌ها با فرمت Parquet ذخیره شده‌اند:

```text
data/processed/
├── observations.parquet
├── species.parquet
└── images.parquet
```

## ساختار دیتاست

دیتاست از سه جدول مرتبط تشکیل شده است.

### ۱. Observations — مشاهدات

**فایل:** `observations.parquet`

هر ردیف نشان‌دهنده یک مشاهده ثبت‌شده از یک حیوان است. این جدول شامل تمام ستون‌های اصلی داده‌های اولیه، به‌علاوه دو ستون جدید است.

| ستون | توضیح |
|---|---|
| `id` | شناسه مشاهده |
| `uuid` | شناسه یکتای مشاهده |
| `observed_on` | تاریخ مشاهده |
| `time_observed_at` | زمان مشاهده |
| `user_id` | شناسه کاربر ثبت‌کننده مشاهده |
| `quality_grade` | کیفیت و اعتبار مشاهده |
| `url` | لینک صفحه مشاهده در iNaturalist |
| `image_url` | لینک تصویر |
| `latitude` | عرض جغرافیایی |
| `longitude` | طول جغرافیایی |
| `taxon_geoprivacy` | وضعیت محرمانگی موقعیت جغرافیایی |
| `common_name` | نام رایج حیوان |
| `taxon_class_name` | رده جانور |
| `taxon_order_name` | راسته جانور |
| `taxon_family_name` | خانواده جانور |
| `taxon_group` | گروه جانوری |
| `species_id` | شناسه گونه مرتبط |

تمام ستون‌های اصلی CSVها حفظ شده‌اند.

---

### ۲. Species — گونه‌ها

**فایل:** `species.parquet`

این جدول اطلاعات طبقه‌بندی گونه‌های موجود در دیتاست را نگهداری می‌کند.

| ستون | توضیح |
|---|---|
| `species_id` | شناسه یکتای ساخته‌شده برای گونه |
| `common_name` | نام رایج |
| `taxon_class_name` | رده |
| `taxon_order_name` | راسته |
| `taxon_family_name` | خانواده |

گونه‌ها بر اساس ترکیب اطلاعات طبقه‌بندی شناسایی شده‌اند؛ زیرا شناسه رسمی taxon در داده‌های اولیه موجود نبوده است.

---

### ۳. Images — تصاویر

**فایل:** `images.parquet`

این جدول اطلاعات تصاویر مربوط به مشاهدات را نگهداری می‌کند.

| ستون | توضیح |
|---|---|
| `image_id` | شناسه تصویر |
| `observation_uuid` | شناسه مشاهده مرتبط |
| `image_path` | مسیر فایل تصویر |
| `animal_present` | وجود یا عدم وجود حیوان در تصویر |

ستون `animal_present` برای برچسب‌گذاری در مراحل بعدی در نظر گرفته شده است و فعلاً مقداردهی نشده است.

---

## ارتباط جداول

ارتباط بین سه جدول به شکل زیر است:

```text
Species
   |
   | species_id
   |
Observations
   |
   | uuid = observation_uuid
   |
Images
```

- هر مشاهده به یک گونه متصل می‌شود.
- هر تصویر به مشاهده مربوط به خودش متصل می‌شود.
- اطلاعات اصلی مشاهده در جدول Observations نگهداری می‌شود.

---

## مراحل پیش‌پردازش انجام‌شده

1. یکپارچه‌سازی چهار فایل CSV مربوط به دوزیستان، پرندگان، پستانداران و خزندگان.
2. حفظ ستون‌های اصلی داده‌ها.
3. ساخت جدول مستقل گونه‌ها و اختصاص شناسه یکتا.
4. ساخت جدول تصاویر و اتصال تصاویر به مشاهدات.
5. شناسایی تصاویر مشابه با استفاده از Perceptual Hash (pHash).
6. حذف تصاویر با شباهت بیشتر از ۹۰ درصد از مجموعه پردازش‌شده.
7. حذف مشاهداتی که تصویر متناظر ندارند.

فایل‌های خام اولیه بدون تغییر باقی مانده‌اند.
```